# Etapa 2 – Limpieza y Transformación (ETL)
Proyecto: Subtes de Buenos Aires

Objetivo: asegurar calidad de datos (nulos, duplicados, tipos, consistencia), transformar variables necesarias y dejar un dataset final listo para visualización (Etapa 3).


In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None) # muestra todas las columnas
pd.set_option("display.width", 120) # ajusto el ancho


## Extract (fuentes)

Utilizo como en la etapa 1 el dataset correspondiente al año 2019 para continuar con el ETL.

In [2]:
path = "../data/raw/"

df_2019 = pd.read_csv(path + "historico_2019.csv")
df_2019.head()

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total
0,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Lima_N_Turn02,Lima,1.0,0.0,0.0,1.0
1,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Loria_N_Turn03,Loria,3.0,0.0,0.0,3.0
2,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Miserere_Q_HALL_Turn01,Plaza Miserere,3.0,0.0,0.0,3.0
3,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Miserere_S_Turn01,Plaza Miserere,6.0,0.0,0.0,6.0
4,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Miserere_S_Turn03,Plaza Miserere,10.0,0.0,0.0,10.0


In [3]:
df_2019.info() # muestro info del dataframe por si hay columnas con muchos nulos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12662343 entries, 0 to 12662342
Data columns (total 11 columns):
 #   Column           Dtype  
---  ------           -----  
 0   periodo          int64  
 1   fecha            object 
 2   desde            object 
 3   hasta            object 
 4   linea            object 
 5   molinete         object 
 6   estacion         object 
 7   pax_pagos        float64
 8   pax_pases_pagos  float64
 9   pax_franq        float64
 10  total            float64
dtypes: float64(4), int64(1), object(6)
memory usage: 1.0+ GB


In [4]:
df_2019.describe() # estadisticas descriptivas del dataframe

,periodo,pax_pagos,pax_pases_pagos,pax_franq,total
count,1.266234e+07,1.266234e+07,1.266234e+07,1.266234e+07,1.266234e+07
mean,2.019066e+05,2.563776e+01,1.186783e-01,1.200694e+00,2.695714e+01
std,3.444613e+00,2.832257e+01,5.024237e-01,4.085881e+00,2.939805e+01
min,2.019010e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.019040e+05,5.000000e+00,0.000000e+00,0.000000e+00,6.000000e+00
50%,2.019070e+05,1.600000e+01,0.000000e+00,0.000000e+00,1.700000e+01
75%,2.019100e+05,3.600000e+01,0.000000e+00,2.000000e+00,3.800000e+01
max,2.019120e+05,4.310000e+02,8.500000e+01,1.243800e+04,1.245900e+04


### Análisis Exploratorio Inicial

**Observaciones clave del dataset 2019:**
- Verificación de estructura y calidad de datos
- Identificación de tipos de datos incorrectos
- Preparación para transformaciones

In [5]:
df_2019.isnull().sum()

periodo            0
fecha              0
desde              0
hasta              0
linea              0
molinete           0
estacion           0
pax_pagos          0
pax_pases_pagos    0
pax_franq          0
total              0
dtype: int64

In [6]:
# Validación de integridad: total debe ser suma de componentes
df_2019['total_calculado'] = (
    df_2019['pax_pagos'] + 
    df_2019['pax_pases_pagos'] + 
    df_2019['pax_franq']
)

inconsistencias = (df_2019['total'] != df_2019['total_calculado']).sum()
print(f"Registros con inconsistencia en 'total': {inconsistencias}")

if inconsistencias > 0:
    print("\nEjemplos de inconsistencias:")
    print(df_2019[df_2019['total'] != df_2019['total_calculado']].head())
    # Corrección si hay inconsistencias
    df_2019['total'] = df_2019['total_calculado']

df_2019.drop('total_calculado', axis=1, inplace=True)

Registros con inconsistencia en 'total': 0


## Insights
Pese a que el dataframe no tenga nulos, es notorio que posee una gran variedad de columnas con tipos de datos erroneos:

fecha    object

desde    object

hasta    object

linea    object

estacion object


In [7]:
# Conversion de fechas
df_2019['fecha'] = pd.to_datetime(df_2019['fecha'], errors='coerce')
# desde y hasta se dejan como string/object ya que representan rangos horarios
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12662343 entries, 0 to 12662342
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   periodo          int64         
 1   fecha            datetime64[ns]
 2   desde            object        
 3   hasta            object        
 4   linea            object        
 5   molinete         object        
 6   estacion         object        
 7   pax_pagos        float64       
 8   pax_pases_pagos  float64       
 9   pax_franq        float64       
 10  total            float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(5)
memory usage: 1.0+ GB


### Conversión y estandarización de campos temporales

La columna `fecha` se convirtió al tipo `datetime` para asegurar una correcta interpretación temporal de los registros y habilitar análisis basados en tiempo (por día, mes o período).

Las columnas `desde` y `hasta` se mantienen como texto ya que representan rangos o franjas horarias en formato string (por ejemplo: "06:00 - 07:00").

El uso del parámetro `errors='coerce'` asegura un manejo controlado de posibles valores mal formateados en la columna `fecha`.

## Optimizacion de memoria

In [8]:
df_2019.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12662343 entries, 0 to 12662342
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   periodo          int64         
 1   fecha            datetime64[ns]
 2   desde            object        
 3   hasta            object        
 4   linea            object        
 5   molinete         object        
 6   estacion         object        
 7   pax_pagos        float64       
 8   pax_pases_pagos  float64       
 9   pax_franq        float64       
 10  total            float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(5)
memory usage: 4.1 GB


In [9]:
df_2019["periodo"] = df_2019["periodo"].astype("int32") # no necesita tanto espacio de memoria como int64

# Optimizar columnas de pasajeros
for col in ["pax_pagos", "pax_pases_pagos", "pax_franq", "total"]:
    max_val = df_2019[col].max()
    if max_val <= 255:
        df_2019[col] = df_2019[col].astype("uint8")
    elif max_val <= 65535:
        df_2019[col] = df_2019[col].astype("uint16")
    else:
        df_2019[col] = df_2019[col].astype("uint32")

for col in ["linea", "molinete", "estacion"]:
    df_2019[col] = df_2019[col].astype("category") # lo mismo aca, convierto a category que ocupa menos espacio de memoria

In [10]:
df_2019.info(memory_usage="deep") #chequeo post optimizacion de memoria


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12662343 entries, 0 to 12662342
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   periodo          int32         
 1   fecha            datetime64[ns]
 2   desde            object        
 3   hasta            object        
 4   linea            category      
 5   molinete         category      
 6   estacion         category      
 7   pax_pagos        uint16        
 8   pax_pases_pagos  uint8         
 9   pax_franq        uint16        
 10  total            uint16        
dtypes: category(3), datetime64[ns](1), int32(1), object(2), uint16(3), uint8(1)
memory usage: 1.6 GB


### Optimización de memoria
Dado el volumen del dataset (millones de filas), decidi optimizar los siguientes tipos de datos:
- `periodo` se convirtió a `int32`.
- `linea`, `molinete` y `estacion` se convirtieron a `category`, reduciendo el uso de memoria y mejorando el rendimiento en agrupaciones.


In [11]:
# Normalización de textos (consistencia semántica)
df_2019["estacion"] = (
    df_2019["estacion"].astype("string")
    .str.strip()
    .str.title()
    .astype("category")
)

df_2019["linea"] = (
    df_2019["linea"].astype("string")
    .str.strip()
    .str.upper()
    .astype("category")
)

df_2019["molinete"] = (
    df_2019["molinete"].astype("string")
    .str.strip()
    .str.upper()
    .astype("category")
)
df_2019.head()

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total
0,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LIMA_N_TURN02,Lima,1,0,0,1
1,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LORIA_N_TURN03,Loria,3,0,0,3
2,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_Q_HALL_TURN01,Plaza Miserere,3,0,0,3
3,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN01,Plaza Miserere,6,0,0,6
4,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN03,Plaza Miserere,10,0,0,10


In [12]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12662343 entries, 0 to 12662342
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   periodo          int32         
 1   fecha            datetime64[ns]
 2   desde            object        
 3   hasta            object        
 4   linea            category      
 5   molinete         category      
 6   estacion         category      
 7   pax_pagos        uint16        
 8   pax_pases_pagos  uint8         
 9   pax_franq        uint16        
 10  total            uint16        
dtypes: category(3), datetime64[ns](1), int32(1), object(2), uint16(3), uint8(1)
memory usage: 471.0+ MB


In [13]:
# Cuenta de duplicados luego de la limpieza
duplicados = df_2019.duplicated().sum()
print(f"Cantidad de filas duplicadas: {duplicados}")

Cantidad de filas duplicadas: 0


In [14]:
# No hace falta porque no hay duplicados pero si hubiese usaria esto:
# df_2019 = df_2019.drop_duplicates().reset_index(drop=True) -> el reset_index es para que no me quede en la data el indice viejo como columna

df_2019.head()

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total
0,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LIMA_N_TURN02,Lima,1,0,0,1
1,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LORIA_N_TURN03,Loria,3,0,0,3
2,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_Q_HALL_TURN01,Plaza Miserere,3,0,0,3
3,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN01,Plaza Miserere,6,0,0,6
4,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN03,Plaza Miserere,10,0,0,10


In [15]:
# Verificación del formato de franjas horarias
print("Formato de 'desde' y 'hasta':")
print(df_2019[['desde', 'hasta']].head(10))

Formato de 'desde' y 'hasta':
      desde     hasta
0  08:00:00  08:15:00
1  08:00:00  08:15:00
2  08:00:00  08:15:00
3  08:00:00  08:15:00
4  08:00:00  08:15:00
5  08:00:00  08:15:00
6  08:00:00  08:15:00
7  08:00:00  08:15:00
8  08:00:00  08:15:00
9  08:00:00  08:15:00


In [16]:
# Variables derivadas para la Etapa 3

# Intenta extraer la hora solo si el formato es correcto
try:
    df_2019["hora_desde"] = pd.to_datetime(
        df_2019["desde"], 
        format='%H:%M:%S', 
        errors='coerce'
    ).dt.hour.astype("Int8")
    
    df_2019["hora_hasta"] = pd.to_datetime(
        df_2019["hasta"], 
        format='%H:%M:%S', 
        errors='coerce'
    ).dt.hour.astype("Int8")
    
    # Verificar si la conversión fue exitosa
    nulos_hora = df_2019[['hora_desde', 'hora_hasta']].isnull().sum()
    print(f"Valores nulos tras conversión:\n{nulos_hora}")
    
except Exception as e:
    print(f"No se pudieron procesar las horas: {e}")
    print("Se mantienen 'desde' y 'hasta' en formato original")

# Variables temporales adicionales
df_2019["dia_semana"] = df_2019["fecha"].dt.day_name()
df_2019["mes"] = df_2019["fecha"].dt.month.astype("Int8")
df_2019["dia_mes"] = df_2019["fecha"].dt.day.astype("Int8")
df_2019["es_fin_semana"] = df_2019["dia_semana"].isin(['Saturday', 'Sunday']).astype("int8")

df_2019.head()

Valores nulos tras conversión:
hora_desde    0
hora_hasta    0
dtype: int64


,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,hora_desde,hora_hasta,dia_semana,mes,dia_mes,es_fin_semana
0,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LIMA_N_TURN02,Lima,1,0,0,1,8,8,Tuesday,1,1,0
1,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LORIA_N_TURN03,Loria,3,0,0,3,8,8,Tuesday,1,1,0
2,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_Q_HALL_TURN01,Plaza Miserere,3,0,0,3,8,8,Tuesday,1,1,0
3,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN01,Plaza Miserere,6,0,0,6,8,8,Tuesday,1,1,0
4,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN03,Plaza Miserere,10,0,0,10,8,8,Tuesday,1,1,0


In [17]:
# Exportación de los datos limpios
path_clean = "../data/clean/"

# Verificación previa
print("Resumen del dataset limpio:")
print(f"- Shape: {df_2019.shape}")
print(f"- Memoria: {df_2019.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"- Columnas: {list(df_2019.columns)}")

df_2019.to_csv(path_clean + "historico_2019_clean.csv", index=False)

# Verificación post-exportación
df_test = pd.read_csv(path_clean + "historico_2019_clean.csv", nrows=5)
print(f"\nDatos exportados correctamente en: {path_clean}")
print(f"Primeras filas verificadas: {len(df_test)} registros")

Resumen del dataset limpio:
- Shape: (12662343, 17)
- Memoria: 2441.45 MB
- Columnas: ['periodo', 'fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'total', 'hora_desde', 'hora_hasta', 'dia_semana', 'mes', 'dia_mes', 'es_fin_semana']

Datos exportados correctamente en: ../data/clean/
Primeras filas verificadas: 5 registros


## Conclusiones de la Etapa 2

**Transformaciones aplicadas:**
- Conversión de tipos de datos (datetime, categorías, optimización numérica)
- Validación de integridad (total = suma de componentes)
- Normalización de textos (mayúsculas/minúsculas consistentes)
- Creación de variables derivadas (hora, día semana, mes, fin de semana)
- Reducción de uso de memoria mediante optimización de tipos

**Dataset resultante:**
- Registros: ~12.6M
- Calidad: sin nulos, sin duplicados, tipos optimizados
- Memoria optimizada para procesamiento eficiente

**Listo para Etapa 3 (Visualización):**
El dataset `historico_2019_clean.csv` está preparado para análisis de:
- Patrones temporales (horarios pico, días semana vs fin de semana)
- Distribución por líneas y estaciones
- Análisis de tipos de pasajeros (pagos vs franquicias)